# Steps 6 & 7 — Benchmark & HPC Optimization

Same cleaning logic across four engines, benchmarked on the same `raw_data.csv`:

1. **pandas** — single-threaded baseline (step 6).
2. **Polars** — vectorised, multi-threaded Rust engine.
3. **Multiprocessing** — pandas split across CPU cores via `Pool.map`.
4. **PySpark** — distributed DataFrame on `local[*]`.

Each engine runs `REPEATS` times; per-run and averaged metrics are saved to `benchmark_runs.csv` / `benchmark_results.csv`, and comparison charts are saved as PNGs.

In [ ]:
import os
import importlib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import cleaners, worker, benchmark_utils
importlib.reload(cleaners)
importlib.reload(worker)
importlib.reload(benchmark_utils)
from cleaners import clean_pandas, clean_polars, clean_pyspark
from worker import clean_multiprocessing
from benchmark_utils import benchmark

RAW = 'raw_data.csv'
RESULTS = 'benchmark_results.csv'
RUNS = 'benchmark_runs.csv'
REPEATS = 5
sns.set_theme(style='whitegrid')

## Load raw data once

In [ ]:
raw_df = pd.read_csv(RAW)
n_rows = len(raw_df)
print(f'rows: {n_rows}')

## Pandas — baseline (Step 6)

In [ ]:
pandas_m = benchmark(clean_pandas, raw_df, repeats=REPEATS, n_rows=n_rows)
pandas_m['engine'] = 'pandas'
print('Per-run metrics:')
display(pandas_m['runs_df'])
print('\nAverage:')
{k: v for k, v in pandas_m.items() if k not in ('runs', 'runs_df')}

## Polars

In [ ]:
polars_m = benchmark(clean_polars, RAW, repeats=REPEATS, n_rows=n_rows)
polars_m['engine'] = 'polars'
print('Per-run metrics:')
display(polars_m['runs_df'])
print('\nAverage:')
{k: v for k, v in polars_m.items() if k not in ('runs', 'runs_df')}

## Multiprocessing

In [ ]:
mp_m = benchmark(clean_multiprocessing, raw_df, repeats=REPEATS, n_rows=n_rows)
mp_m['engine'] = 'multiprocessing'
print('Per-run metrics:')
display(mp_m['runs_df'])
print('\nAverage:')
{k: v for k, v in mp_m.items() if k not in ('runs', 'runs_df')}

## PySpark
Requires Java + `winutils` on Windows. If this cell errors, install Java 11/17 and set `JAVA_HOME`, then re-run.

In [ ]:
spark_m = benchmark(clean_pyspark, RAW, repeats=REPEATS, n_rows=n_rows)
spark_m['engine'] = 'pyspark'
print('Per-run metrics:')
display(spark_m['runs_df'])
print('\nAverage:')
{k: v for k, v in spark_m.items() if k not in ('runs', 'runs_df')}

## Persist results (averages + per-run)

In [ ]:
cols = ['engine', 'wall_time_s', 'cpu_pct', 'peak_mem_mb', 'rss_mem_mb', 'rows_out', 'throughput_rps']

def _scalar(m):
    return {k: v for k, v in m.items() if k not in ('runs', 'runs_df')}

engines = (pandas_m, polars_m, mp_m, spark_m)

results = pd.DataFrame([_scalar(m) for m in engines])[cols]
results.to_csv(RESULTS, index=False)

per_run = pd.concat([m['runs_df'].assign(engine=m['engine']) for m in engines], ignore_index=True)
per_run = per_run[['engine', 'run', 'wall_time_s', 'cpu_pct', 'peak_mem_mb', 'rss_mem_mb', 'rows_out', 'throughput_rps']]
per_run.to_csv(RUNS, index=False)

print('Per-run results:')
display(per_run)
print('\nAveraged results:')
results

## Visualisations

In [ ]:
def bar(metric, ylabel, fname, palette='viridis'):
    fig, ax = plt.subplots(figsize=(7, 4.2))
    sns.barplot(data=results, x='engine', y=metric, ax=ax, palette=palette)
    ax.set_title(f'{ylabel} by engine')
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.show()

bar('wall_time_s', 'Wall time (s)', 'fig_time.png', 'rocket')
bar('throughput_rps', 'Throughput (rows/s)', 'fig_throughput.png', 'mako')
bar('peak_mem_mb', 'Peak memory (MB)', 'fig_memory.png', 'crest')

In [ ]:
# Speedup vs pandas baseline
base = results.loc[results['engine'] == 'pandas', 'wall_time_s'].iloc[0]
speedup = results.assign(speedup=base / results['wall_time_s'])[['engine', 'wall_time_s', 'speedup']]
speedup

## Interpretation
- **Polars** typically delivers the largest single-machine speedup thanks to Rust + multi-threaded execution with low overhead.
- **Multiprocessing** scales pandas across cores but pays serialization cost for the chunk transfer; gains are real but smaller than Polars.
- **PySpark** shines on much larger datasets; on ~290k rows the JVM startup and task overhead can make it *slower* than the baseline. Note that `tracemalloc` does not see JVM memory, so the peak-memory bar understates Spark's true footprint.